# Lab 2a · Leer la oferta: lo exacto, en código

Hoy construyes **tu agente de candidaturas**. Le pasas la oferta que me mandaste por email y la lee, adapta tu CV
**sin inventar nada** a través de un servidor MCP, investiga la empresa con fuentes y te prepara un plan de cuatro
semanas y tu pitch de tres frases. Al final, todo en una **webapp**.

| Lab | Pieza | Tiempo |
|---|---|---|
| **2a** | Leer la oferta (robots.txt y JSON-LD) y estructurarla con el modelo | 15' |
| **2b** | Tu CV detrás de un servidor MCP, con un guardarraíl que no deja inventar | 20' |
| **2c** | Investigar la empresa con fuentes… y lanzar tu webapp | 15' |

Lo que escribas en las celdas ✏️ se guarda en `sesion2/mis_piezas/` y **lo usará tu webapp**. ¿Atascado más de 3
minutos? `mucdia.lab.pista("jobposting")` o `mucdia.lab.solucion("jobposting")` (también `"oferta"`).

In [ ]:
import sys

import pandas as pd

import mucdia

sys.path.insert(0, str(mucdia.config.RAIZ / "sesion2"))
import candidatura as C  # noqa: E402

PIEZAS = C.piezas.CARPETA      # tus celdas ✏️ guardan aquí su código: lo usará tu webapp
PIEZAS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 110)
cliente = mucdia.cliente()

MI_OFERTA = ""  # ✏️ pega entre las comillas el enlace de la oferta que me mandaste por email
OFERTA = C.oferta.elegir(MI_OFERTA)

## Paso 1 · ¿Me dejas leerte? (robots.txt)

Cada web publica en `/robots.txt` qué páginas pueden leer los programas (el estándar es el RFC 9309). Nuestro
agente lo respeta: si una web lo prohíbe, **no la lee** y te pide que pegues el texto. Y si la web pone un muro
(captcha, «activa JavaScript»), tampoco insiste.

In [ ]:
for url in ["https://es.linkedin.com/jobs/view/data-analyst-at-insud-pharma-4489730941",
            "https://www.infojobs.net/madrid/cientifico-datos/of-i127b0a245c4e59a03c9c5c73fdfac3",
            OFERTA]:
    ok, motivo = C.web.permitido(url)
    print(f"{'✅' if ok else '🚫'} {url[:75]}…\n   {motivo}")

🤔 LinkedIn lo prohíbe a todos los programas (y sus condiciones de uso, también). InfoJobs lo permite en
robots.txt… pero enseña un captcha a quien no es un navegador. Las **APIs públicas** (Wikipedia, por ejemplo) son
otra cosa: tienen sus propias condiciones de uso, pensadas para programas, y las cumplimos (lab 2c).

## Paso 2 · ✏️ El atajo de los buscadores: JSON-LD

Muchas webs de empleo incrustan la oferta en un bloque invisible para que Google la indexe: un JSON con el
vocabulario de [schema.org](https://schema.org/JobPosting). Si está, no hace falta adivinar dónde está cada cosa en
el HTML:

```html
<script type="application/ld+json">
{"@context": "https://schema.org", "@type": "JobPosting",
 "title": "Data Analyst Junior",
 "hiringOrganization": {"@type": "Organization", "name": "Ejemplo S.A."},
 "jobLocation": {"@type": "Place", "address": {"addressLocality": "Madrid"}},
 "datePosted": "2026-11-20", "description": "<p>Buscamos…</p>"}
</script>
```

Escribe `extraer_jobposting(html)`: devuelve el primer objeto `JobPosting` o `None`. Ojo: un bloque puede ser un
objeto, una **lista** de objetos o un objeto con `"@graph"` (otra lista), y alguno puede no ser JSON válido.

In [ ]:
%%writefile "{PIEZAS}/oferta.py"
import json

from bs4 import BeautifulSoup


def extraer_jobposting(html: str) -> dict | None:
    """Devuelve el primer objeto JobPosting de los bloques JSON-LD del HTML, o None si no hay."""
    sopa = BeautifulSoup(html or "", "html.parser")
    for script in sopa.find_all("script", type="application/ld+json"):
        # ✏️ TODO 1: convierte script.string en un objeto de Python con json.loads(..., strict=False);
        #            si no es JSON válido (json.JSONDecodeError), salta al siguiente bloque con continue
        # ✏️ TODO 2: el resultado puede ser un dict, una lista de dicts o un dict con "@graph" (otra lista):
        #            recórrelos y devuelve el primero cuyo "@type" sea "JobPosting"
        pass
    return None

In [ ]:
extraer_jobposting = C.piezas.cargar("oferta", "extraer_jobposting")
mucdia.lab.comprobar_jobposting(extraer_jobposting)

## Paso 3 · Lee TU oferta

`C.oferta.leer_oferta(url)` prueba, por este orden: robots.txt → el JSON público de Workday (BBVA, AstraZeneca y
muchas grandes empresas pintan su web de empleo con JavaScript, pero cargan la oferta de un JSON) → **tu** lector
de JSON-LD → microdatos de schema.org → el texto de la página.

In [ ]:
pagina = C.oferta.leer_oferta(OFERTA)
if pagina["ok"]:
    print(f"✅ Leída de: {pagina['fuente']} · {C.fmt(len(pagina['texto']))} caracteres · robots.txt: {pagina['robots']}\n")
    print(pagina["texto"][:1200], "…")
else:
    print(f"🚫 {pagina['mensaje']}\n👉 {pagina['consejo']}")

In [ ]:
TEXTO_OFERTA = """"""  # ✏️ solo si tu web no deja leerla: pega entre las comillas el texto de la oferta
if TEXTO_OFERTA.strip():
    pagina = C.oferta.desde_texto(TEXTO_OFERTA, url=OFERTA)
    print(f"✅ Usaré el texto que has pegado ({C.fmt(len(pagina['texto']))} caracteres)")

## Paso 4 · ✏️ El juicio, en el modelo: estructurar la oferta

El texto ya está; ahora hace falta **juicio**: ¿qué es imprescindible y qué solo se valora? ¿Qué palabras buscará el
filtro automático (ATS)? Se lo pedimos al modelo con un **esquema** (Pydantic). Las descripciones de los campos son
sus instrucciones, campo a campo: complétalas.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Requisito(BaseModel):
    texto: str = Field(description="El requisito, resumido en una línea y fiel a la oferta")
    tipo: Literal["imprescindible", "deseable"] = Field(
        description="✏️ TODO: ¿cuándo es imprescindible y cuándo deseable? (¿qué palabras de la oferta lo indican?)")
    categoria: Literal["técnica", "negocio", "idioma", "formación", "experiencia", "personal"] = Field(
        description="técnica: herramientas y métodos; negocio: sector o función; idioma; formación: títulos; "
                    "experiencia: años o puestos previos; personal: competencias transversales")


class Oferta(BaseModel):
    empresa: str = Field(description="Nombre comercial de la empresa que contrata (no el del portal de empleo)")
    puesto: str = Field(description="Nombre del puesto tal como lo anuncia la oferta")
    ubicacion: str = Field(description="Ciudad o ciudades; «no consta» si no lo dice")
    modalidad: Literal["presencial", "híbrido", "remoto", "no consta"] = Field(description="Modalidad de trabajo")
    nivel: Literal["prácticas", "júnior", "intermedio", "sénior", "no consta"] = Field(
        description="Nivel según la experiencia que pide: prácticas; júnior (0-2 años); intermedio (3-5); sénior (más)")
    salario: str = Field(description="✏️ TODO: ¿y si la oferta no lo dice? (que no se lo invente)")
    resumen: str = Field(description="Qué hará la persona y para qué, en dos frases")
    tareas: list[str] = Field(description="Las funciones principales del puesto, una por línea")
    requisitos: list[Requisito] = Field(description="Todos los requisitos de la oferta, imprescindibles y deseables")
    palabras_clave: list[str] = Field(description="✏️ TODO: ¿qué palabras buscaría un filtro automático (ATS) en tu CV?")

In [ ]:
mucdia.lab.comprobar_oferta(Oferta)
oferta = await C.oferta.analizar_oferta(cliente, pagina, esquema=Oferta)
C.oferta.guardar_oferta(oferta, pagina)    # para los labs 2b y 2c
print(f"{oferta.puesto} · {oferta.empresa} · {oferta.nivel} · {oferta.modalidad} · salario: {oferta.salario}")
print(oferta.resumen, "\n")
requisitos = pd.DataFrame([r.model_dump() for r in oferta.requisitos])
print(f"📏 {len(requisitos)} requisitos ({(requisitos.tipo == 'imprescindible').sum()} imprescindibles) · "
      f"{len(oferta.palabras_clave)} palabras clave: {', '.join(oferta.palabras_clave)}")
requisitos

🤔 Para pensar:
- ¿Qué ha hecho el **código** (sin equivocarse nunca) y qué el **modelo** (con juicio… y algún error)?
- Compara con lo que habríais sacado vosotros a mano. ¿Te fías de su
  clasificación imprescindible/deseable? ¿Cómo lo comprobarías con 20 ofertas en vez de con una?

## ⭐ Para ir más allá

1. Lee una oferta de Workday: `C.oferta.leer_oferta("https://bbva.wd3.myworkdayjobs.com/…")`. Mira en
   `candidatura/oferta.py` cómo `_workday()` cambia la URL de la página por la del JSON que carga la propia web.
2. Prueba con una oferta en inglés: ¿mantiene los nombres de las herramientas y responde en español?